In [ ]:
# Setup imports and helper functions
import os, sys
proj_root = os.path.abspath(os.path.join('..', "lib"))
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)

import cv2, numpy as np, matplotlib.pyplot as plt
from face_detector import FaceDetector, select_image_file
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd

detector = FaceDetector()

def cosine_sim(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if a.ndim == 1:
        a = a.reshape(1, -1)
    if b.ndim == 1:
        b = b.reshape(1, -1)
    dot = np.dot(a, b.T)
    na = np.linalg.norm(a, axis=1)
    nb = np.linalg.norm(b, axis=1)
    return (dot / (na[:,None] * nb[None,:] + 1e-8))



In [ ]:

# Select images
GALLERY_IMAGE = None  # image with known faces (can be a group photo)
QUERY_IMAGE = None    # image with faces to match against gallery

if GALLERY_IMAGE is None:
    print('Select GALLERY_IMAGE (knowns)')
    GALLERY_IMAGE = select_image_file()
if QUERY_IMAGE is None:
    print('Select QUERY_IMAGE (queries)')
    QUERY_IMAGE = select_image_file()

if not GALLERY_IMAGE or not QUERY_IMAGE:
    raise RuntimeError('Both gallery and query images are required.')


In [ ]:

# Load, detect, and visualize
g_img = cv2.imread(GALLERY_IMAGE)
q_img = cv2.imread(QUERY_IMAGE)
g_faces = detector.process_frame(g_img)
q_faces = detector.process_frame(q_img)
print(f'Gallery faces: {len(g_faces)} | Query faces: {len(q_faces)}')

# Convert BGR to RGB for matplotlib
g_img_rgb = cv2.cvtColor(g_img, cv2.COLOR_BGR2RGB)
q_img_rgb = cv2.cvtColor(q_img, cv2.COLOR_BGR2RGB)

# Create side-by-side visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot gallery image
ax1.imshow(g_img_rgb)
ax1.set_title(f'Gallery Image: {os.path.basename(GALLERY_IMAGE)}\nFaces Detected: {len(g_faces)}', fontsize=12, fontweight='bold')
ax1.axis('off')

# Draw bounding boxes on gallery image
for i, face in enumerate(g_faces):
    bbox = face['bbox']
    cv2.rectangle(g_img_rgb, 
                 (int(bbox[0]), int(bbox[1])), 
                 (int(bbox[2]), int(bbox[3])), 
                 (0, 255, 0), 2)
    cv2.putText(g_img_rgb, f'G{i+1}', 
               (int(bbox[0]), int(bbox[1]-10)), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

# Plot query image  
ax2.imshow(q_img_rgb)
ax2.set_title(f'Query Image: {os.path.basename(QUERY_IMAGE)}\nFaces Detected: {len(q_faces)}', fontsize=12, fontweight='bold')
ax2.axis('off')

# Draw bounding boxes on query image
for i, face in enumerate(q_faces):
    bbox = face['bbox']
    cv2.rectangle(q_img_rgb, 
                 (int(bbox[0]), int(bbox[1])), 
                 (int(bbox[2]), int(bbox[3])), 
                 (255, 0, 0), 2)
    cv2.putText(q_img_rgb, f'Q{i+1}', 
               (int(bbox[0]), int(bbox[1]-10)), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

# Update the images with bounding boxes
ax1.imshow(g_img_rgb)
ax2.imshow(q_img_rgb)

plt.tight_layout()
plt.show()

# Print detailed face information
print(f"\nGallery Faces Details:")
for i, face in enumerate(g_faces):
    bbox = face['bbox']
    print(f"  G{i+1}: BBox [{bbox[0]:.1f}, {bbox[1]:.1f}, {bbox[2]:.1f}, {bbox[3]:.1f}]")

print(f"\nQuery Faces Details:")
for i, face in enumerate(q_faces):
    bbox = face['bbox']
    print(f"  Q{i+1}: BBox [{bbox[0]:.1f}, {bbox[1]:.1f}, {bbox[2]:.1f}, {bbox[3]:.1f}]")



In [ ]:
# Collect embeddings and crops
g_embeddings = [f['embedding'].astype(np.float32) for f in g_faces if f.get('embedding') is not None]
g_crops = [cv2.cvtColor(f['face_img'], cv2.COLOR_BGR2RGB) for f in g_faces]
q_embeddings = [f['embedding'].astype(np.float32) for f in q_faces if f.get('embedding') is not None]
q_crops = [cv2.cvtColor(f['face_img'], cv2.COLOR_BGR2RGB) for f in q_faces]

if not g_embeddings or not q_embeddings:
    raise RuntimeError('Embeddings missing for gallery or query faces. Ensure InsightFace recognition models are loaded.')

# Compute similarity matrix
sim = cosine_sim(np.stack(q_embeddings), np.stack(g_embeddings))  # shape (Q, G)
print('Similarity matrix shape:', sim.shape)

# NEW: Visualize embedding vectors using dimensionality reduction
all_embeddings = np.vstack([np.stack(g_embeddings), np.stack(q_embeddings)])
labels = ['Gallery'] * len(g_embeddings) + ['Query'] * len(q_embeddings)

# PCA visualization
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(all_embeddings)

plt.figure(figsize=(10, 8))
for i, label in enumerate(['Gallery', 'Query']):
    indices = [j for j, l in enumerate(labels) if l == label]
    plt.scatter(embeddings_2d[indices, 0], embeddings_2d[indices, 1], label=label, alpha=0.7)
    
# Add face images as annotations
for i, (x, y) in enumerate(embeddings_2d):
    if i < len(g_embeddings):
        plt.annotate(f"G{i+1}", (x, y), fontsize=8)
    else:
        plt.annotate(f"Q{i-len(g_embeddings)+1}", (x, y), fontsize=8)

plt.title('Face Embeddings Visualized with PCA')
plt.legend()
plt.grid(True)
plt.show()

# t-SNE visualization (if we have enough samples)
if len(all_embeddings) > 3:
    tsne = TSNE(n_components=2, random_state=42)
    embeddings_tsne = tsne.fit_transform(all_embeddings)
    
    plt.figure(figsize=(10, 8))
    for i, label in enumerate(['Gallery', 'Query']):
        indices = [j for j, l in enumerate(labels) if l == label]
        plt.scatter(embeddings_tsne[indices, 0], embeddings_tsne[indices, 1], label=label, alpha=0.7)
        
    # Add face images as annotations
    for i, (x, y) in enumerate(embeddings_tsne):
        if i < len(g_embeddings):
            plt.annotate(f"G{i+1}", (x, y), fontsize=8)
        else:
            plt.annotate(f"Q{i-len(g_embeddings)+1}", (x, y), fontsize=8)
    
    plt.title('Face Embeddings Visualized with t-SNE')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:

# NEW: Visualize cosine similarity as a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(sim, annot=True, cmap='viridis', 
            xticklabels=[f"G{i+1}" for i in range(len(g_embeddings))],
            yticklabels=[f"Q{i+1}" for i in range(len(q_embeddings))])
plt.title('Cosine Similarity Matrix (Query vs Gallery)')
plt.xlabel('Gallery Faces')
plt.ylabel('Query Faces')
plt.show()

# NEW: Create correlation table between all profiles
# Compute all pairwise similarities
all_sim = cosine_sim(all_embeddings, all_embeddings)

# Create a DataFrame for better visualization
face_labels = [f"G{i+1}" for i in range(len(g_embeddings))] + [f"Q{i+1}" for i in range(len(q_embeddings))]
sim_df = pd.DataFrame(all_sim, index=face_labels, columns=face_labels)

# Display the correlation table
print("Correlation Table Between All Faces:")
print(sim_df)

# Visualize the full correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(sim_df, annot=True, cmap='viridis')
plt.title('Cosine Similarity Matrix (All Faces)')
plt.show()


In [ ]:

# Visualize best matches for each query face
THRESH = 0.35  # similarity threshold for declaring a match (tune for your model)

for qi in range(sim.shape[0]):
    best_idx = int(np.argmax(sim[qi]))
    best_score = float(sim[qi, best_idx])
    matched = best_score >= THRESH
    fig, axs = plt.subplots(1,2, figsize=(8,4))
    axs[0].imshow(q_crops[qi]); axs[0].set_title(f'Query #{qi+1}'); axs[0].axis('off')
    axs[1].imshow(g_crops[best_idx])
    axs[1].set_title(f'Gallery #{best_idx+1}\nScore: {best_score:.3f} -> {"MATCH" if matched else "UNFAMILIAR"}')
    axs[1].axis('off')
    plt.show()

In [ ]:
# Find the best matching pair and worst matching pair
best_match_score = -1
best_match_pair = None
worst_match_score = 2  # cosine similarity ranges from -1 to 1
worst_match_pair = None

# Find best and worst matches
for qi in range(sim.shape[0]):
    for gi in range(sim.shape[1]):
        score = sim[qi, gi]
        if score > best_match_score:
            best_match_score = score
            best_match_pair = (qi, gi, score)
        if score < worst_match_score:
            worst_match_score = score
            worst_match_pair = (qi, gi, score)

print(f"Best match: Q{best_match_pair[0]+1} vs G{best_match_pair[1]+1} (score: {best_match_pair[2]:.3f})")
print(f"Worst match: Q{worst_match_pair[0]+1} vs G{worst_match_pair[1]+1} (score: {worst_match_score:.3f})")

# Create a 2x2 figure to combine both visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Best matching pair - Vector encodings
qi_best, gi_best, score_best = best_match_pair
query_vec_best = q_embeddings[qi_best]
gallery_vec_best = g_embeddings[gi_best]

axes[0, 0].plot(query_vec_best, 'b-', alpha=0.7, linewidth=2, label=f'Q{qi_best+1}')
axes[0, 0].plot(gallery_vec_best, 'g-', alpha=0.7, linewidth=2, label=f'G{gi_best+1}')
axes[0, 0].fill_between(range(len(query_vec_best)), query_vec_best, gallery_vec_best, alpha=0.3, color='gray', label='Difference')
axes[0, 0].set_title(f'Best Match: Q{qi_best+1} vs G{gi_best+1}\nVector Encodings', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Embedding Dimension')
axes[0, 0].set_ylabel('Activation Value')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Worst matching pair - Vector encodings
qi_worst, gi_worst, score_worst = worst_match_pair
query_vec_worst = q_embeddings[qi_worst]
gallery_vec_worst = g_embeddings[gi_worst]

axes[0, 1].plot(query_vec_worst, 'b-', alpha=0.7, linewidth=2, label=f'Q{qi_worst+1}')
axes[0, 1].plot(gallery_vec_worst, 'g-', alpha=0.7, linewidth=2, label=f'G{gi_worst+1}')
axes[0, 1].fill_between(range(len(query_vec_worst)), query_vec_worst, gallery_vec_worst, alpha=0.3, color='gray', label='Difference')
axes[0, 1].set_title(f'Worst Match: Q{qi_worst+1} vs G{gi_worst+1}\nVector Encodings', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Embedding Dimension')
axes[0, 1].set_ylabel('Activation Value')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Residual comparison
residual_best = np.abs(query_vec_best - gallery_vec_best)
residual_worst = np.abs(query_vec_worst - gallery_vec_worst)

axes[1, 0].plot(residual_best, 'purple', alpha=0.8, linewidth=2, label=f'Best Match Residual')
axes[1, 0].plot(residual_worst, 'orange', alpha=0.8, linewidth=2, label=f'Worst Match Residual')
axes[1, 0].set_title('Absolute Residuals Comparison\n(Difference Between Vectors)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Dimension Index')
axes[1, 0].set_ylabel('Absolute Difference')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Add statistics to residual plot
axes[1, 0].text(0.02, 0.98, f'Best Match Stats:\nMean: {np.mean(residual_best):.4f}\nStd: {np.std(residual_best):.4f}', 
                transform=axes[1, 0].transAxes, fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle="round,pad=0.3", facecolor='purple', alpha=0.3))
axes[1, 0].text(0.02, 0.75, f'Worst Match Stats:\nMean: {np.mean(residual_worst):.4f}\nStd: {np.std(residual_worst):.4f}', 
                transform=axes[1, 0].transAxes, fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle="round,pad=0.3", facecolor='orange', alpha=0.3))

# Plot 4: Similarity scores and metrics comparison
x_pos = [0, 1]
scores = [score_best, score_worst]
residuals = [np.mean(residual_best), np.mean(residual_worst)]
euclidean_dists = [np.linalg.norm(query_vec_best - gallery_vec_best), 
                   np.linalg.norm(query_vec_worst - gallery_vec_worst)]

# Create twin axes for different metrics
ax4a = axes[1, 1]
ax4b = ax4a.twinx()

# Bar width and positions
bar_width = 0.25
x1 = np.array(x_pos) - bar_width
x2 = np.array(x_pos)
x3 = np.array(x_pos) + bar_width

# Plot different metrics
bars1 = ax4a.bar(x1, scores, bar_width, color=['lime', 'red'], alpha=0.8, label='Cosine Similarity')
bars2 = ax4a.bar(x2, residuals, bar_width, color=['purple', 'orange'], alpha=0.8, label='Mean Residual')
bars3 = ax4b.bar(x3, euclidean_dists, bar_width, color=['blue', 'brown'], alpha=0.8, label='Euclidean Dist')

ax4a.set_ylabel('Cosine Similarity & Residuals', color='black')
ax4a.set_ylim(0, max(max(scores), max(residuals)) * 1.3)
ax4b.set_ylabel('Euclidean Distance', color='black')
ax4b.set_ylim(0, max(euclidean_dists) * 1.3)

ax4a.set_xticks(x_pos)
ax4a.set_xticklabels(['Best Match', 'Worst Match'])
ax4a.grid(True, alpha=0.3)

# Add value annotations
for i, (score, residual, euc_dist) in enumerate(zip(scores, residuals, euclidean_dists)):
    ax4a.text(x1[i], score + 0.02, f'{score:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    ax4a.text(x2[i], residual + 0.02, f'{residual:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    ax4b.text(x3[i], euc_dist + max(euclidean_dists)*0.05, f'{euc_dist:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

axes[1, 1].set_title('Multiple Similarity Metrics Comparison', fontsize=12, fontweight='bold')

# Combine legends
lines1, labels1 = ax4a.get_legend_handles_labels()
lines2, labels2 = ax4b.get_legend_handles_labels()
axes[1, 1].legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

# Additional detailed analysis
print(f"\n📊 Detailed Analysis:")
print(f"Best Match (Q{qi_best+1} vs G{gi_best+1}):")
print(f"  Cosine Similarity: {score_best:.4f}")
print(f"  Mean Absolute Difference: {residuals[0]:.4f}")
print(f"  Euclidean Distance: {euclidean_dists[0]:.4f}")
print(f"  Residual Std: {np.std(residual_best):.4f}")

print(f"\nWorst Match (Q{qi_worst+1} vs G{gi_worst+1}):")
print(f"  Cosine Similarity: {score_worst:.4f}")
print(f"  Mean Absolute Difference: {residuals[1]:.4f}")
print(f"  Euclidean Distance: {euclidean_dists[1]:.4f}")
print(f"  Residual Std: {np.std(residual_worst):.4f}")

# Show the actual face images for reference
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Best match faces
axes[0, 0].imshow(q_crops[qi_best])
axes[0, 0].set_title(f'Query Q{qi_best+1} (Best Match)', fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(g_crops[gi_best])
axes[0, 1].set_title(f'Gallery G{gi_best+1} (Best Match)\nScore: {score_best:.3f}', fontweight='bold')
axes[0, 1].axis('off')

# Worst match faces
axes[1, 0].imshow(q_crops[qi_worst])
axes[1, 0].set_title(f'Query Q{qi_worst+1} (Worst Match)', fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(g_crops[gi_worst])
axes[1, 1].set_title(f'Gallery G{gi_worst+1} (Worst Match)\nScore: {score_worst:.3f}', fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Additional: Show correlation between different metrics
print(f"\n🔍 Metric Correlations:")
print(f"Cosine Similarity vs Mean Residual: {np.corrcoef(scores, residuals)[0,1]:.3f}")
print(f"Cosine Similarity vs Euclidean Distance: {np.corrcoef(scores, euclidean_dists)[0,1]:.3f}")
print(f"Mean Residual vs Euclidean Distance: {np.corrcoef(residuals, euclidean_dists)[0,1]:.3f}")